In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
df = pd.read_csv('hotel_bookings.csv')

In [ ]:
df.head(10)
df.tail(10)
df.shape
df.columns
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(exclude=np.number).columns
df.memory_usage(deep=True).sum()

In [ ]:
df.info()
df.describe()
df.describe(include='object')
df.dtypes

In [ ]:
missing = pd.DataFrame({
    'Missing': df.isnull().sum(),
    'Percent': (df.isnull().sum()/len(df))*100
}).sort_values('Missing', ascending=False)
missing
sns.heatmap(df.isnull(), cbar=False)
plt.show()
missing['Missing'].plot(kind='bar')
plt.show()

In [ ]:
duplicates = df.duplicated().sum()
duplicates
df = df.drop_duplicates()

In [ ]:
unique_df = pd.DataFrame({
    'Unique': df.nunique(),
    'Mode': [df[c].mode().iloc[0] if not df[c].mode().empty else np.nan for c in df.columns],
    'ModePercent': [round(df[c].value_counts(normalize=True).iloc[0]*100,2) for c in df.columns]
})
unique_df

In [ ]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

In [ ]:
for col in num_cols:
    print(col)
    print(df[col].describe())
    print(df[col].skew())
    sns.histplot(df[col], kde=True)
    plt.show()
    sns.boxplot(x=df[col])
    plt.show()

In [ ]:
for col in cat_cols:
    print(df[col].value_counts())
    df[col].value_counts().head(15).plot(kind='bar')
    plt.show()
    df[col].value_counts().head(10).plot(kind='pie', autopct='%1.1f%%')
    plt.ylabel('')
    plt.show()

In [ ]:
sns.countplot(x='is_canceled', data=df)
plt.show()

df['is_canceled'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.ylabel('')
plt.show()

In [ ]:
monthly = df.groupby('arrival_date_month').size()
monthly.plot(kind='bar')
plt.show()

yearly = df.groupby('arrival_date_year').size()
yearly.plot(kind='bar')
plt.show()

In [ ]:
iqr_data = []
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3-q1
    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr
    outliers = ((df[col]<lower)|(df[col]>upper)).sum()
    iqr_data.append([col,outliers])
    sns.boxplot(x=df[col])
    plt.show()
pd.DataFrame(iqr_data, columns=['Column','Outliers'])

In [ ]:
z_result = {}
for col in num_cols:
    z = np.abs(zscore(df[col]))
    z_result[col] = (z>3).sum()
pd.DataFrame(z_result.items(), columns=['Column','Z_Outliers'])

In [ ]:
winsor_df = df.copy()
for col in num_cols:
    low = winsor_df[col].quantile(0.01)
    high = winsor_df[col].quantile(0.99)
    winsor_df[col] = winsor_df[col].clip(low, high)

In [ ]:
df['TotalGuests'] = df['adults'] + df['children'].fillna(0) + df['babies']
df['StayDuration'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['BookingSeason'] = np.where(df['arrival_date_month'].isin(['December','January','February'],'Winter',
                              np.where(df['arrival_date_month'].isin(['June','July','August'],'Summer','Monsoon')))
df['LeadTimeCategory'] = pd.cut(df['lead_time'], bins=[-1,30,90,365,1000], labels=['Short','Medium','Long','VeryLong'])
df['GuestType'] = np.where(df['TotalGuests']==1,'Solo',np.where(df['TotalGuests']==2,'Couple','Family'))
df['BookingRiskScore'] = df['lead_time'] + df['previous_cancellations']
df['RevenueOpportunityScore'] = df['StayDuration'] * df['adr']
df['HighValueCustomer'] = np.where(df['adr']>df['adr'].quantile(0.75),1,0)
df['AdvanceBookingGroup'] = pd.qcut(df['lead_time'],4,labels=['Q1','Q2','Q3','Q4'])
df['SpecialRequestCategory'] = pd.cut(df['total_of_special_requests'],bins=[-1,1,3,10],labels=['Low','Medium','High'])

In [ ]:
corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(15,10))
sns.heatmap(corr)
plt.show()

In [ ]:
sns.boxplot(x='hotel', y='adr', data=df)
plt.show()

sns.violinplot(x='customer_type', y='StayDuration', data=df)
plt.show()

In [ ]:
pd.crosstab(df['is_canceled'], df['deposit_type'])
sns.heatmap(pd.crosstab(df['is_canceled'], df['deposit_type']), annot=True)
plt.show()

In [ ]:
df['EstimatedRevenue'] = df['adr'] * df['StayDuration']

df.groupby('hotel')['EstimatedRevenue'].sum().sort_values(ascending=False)
df.groupby('arrival_date_month')['EstimatedRevenue'].sum().sort_values(ascending=False)
df.groupby('market_segment')['EstimatedRevenue'].sum().sort_values(ascending=False)

In [ ]:
standard = pd.DataFrame(StandardScaler().fit_transform(df[num_cols]), columns=num_cols)
minmax = pd.DataFrame(MinMaxScaler().fit_transform(df[num_cols]), columns=num_cols)
robust = pd.DataFrame(RobustScaler().fit_transform(df[num_cols]), columns=num_cols)

standard.describe()
minmax.describe()
robust.describe()

In [ ]:
skew_before = df[num_cols].skew()
log_skew = np.log1p(df[num_cols]).skew()
sqrt_skew = np.sqrt(df[num_cols]).skew()

pd.DataFrame({
    'Before': skew_before,
    'Log': log_skew,
    'Sqrt': sqrt_skew
})

In [ ]:
rules = {
    'ADR Null': df['adr'].isnull().sum(),
    'Hotel Null': df['hotel'].isnull().sum(),
    'Negative LeadTime': (df['lead_time']<0).sum(),
    'Negative ADR': (df['adr']<0).sum(),
    'Negative Adults': (df['adults']<0).sum(),
    'Guests <=0': ((df['adults']+df['children'].fillna(0)+df['babies'])<=0).sum(),
    'ADR <=0': (df['adr']<=0).sum(),
    'StayDuration <=0': (df['StayDuration']<=0).sum()
}
rules

In [ ]:
drop_cols = ['reservation_status_date']
encode_cols = list(df.select_dtypes(exclude=np.number).columns)
scale_cols = list(df.select_dtypes(include=np.number).columns)

print(drop_cols)
print(encode_cols)
print(scale_cols)

In [ ]:
df.to_csv('cleaned_hotel_bookings.csv', index=False)

In [ ]:
class HotelBookingAnalyzer:

    def __init__(self, df):
        self.df = df

    def summary(self):
        return self.df.describe(include='all')

    def missing_values(self):
        return self.df.isnull().sum()

    def duplicates(self):
        return self.df.duplicated().sum()

    def outliers(self):
        result = {}
        for col in self.df.select_dtypes(include=np.number).columns:
            q1 = self.df[col].quantile(0.25)
            q3 = self.df[col].quantile(0.75)
            iqr = q3 - q1
            result[col] = ((self.df[col] < q1-1.5*iqr) | (self.df[col] > q3+1.5*iqr)).sum()
        return result